Data Scraping and Storage

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time
import csv
    
# Set up WebDriver
driver = webdriver.Chrome()
driver.maximize_window()

# IMDb 2024 movie URL
url = "https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31"

movie_data = []
target_movies = 1200  

def scrape_current_page():
    """Scrape all movies on the current page."""
    movies = driver.find_elements(By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item')
    
    for movie in movies:
        try:
            # Movie title
            title_elem = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
            title = title_elem.text
            if '.' in title.split()[0]:
                title = ' '.join(title.split()[1:])

            # Description
            try:
                desc = movie.find_element(By.CSS_SELECTOR, "div.ipc-html-content-inner-div").text
            except:
                desc = "N/A"

            # Rating (may not be available)
            try:
                rating = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star--rating").text
            except:
                rating = "N/A"

            # Vote count
            try:
                votes = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star--voteCount").text.strip('()')
            except:
                votes = "N/A"

            # Image (get URL, not text)
            try:
                image_elem = movie.find_element(By.CSS_SELECTOR, "img.ipc-image")
                image_url = image_elem.get_attribute("src")
            except:
                image_url = "N/A"

            # Duration and Year from metadata
            duration = "N/A"
            year = "N/A"
            try:
                metadata_items = movie.find_elements(By.CSS_SELECTOR, "span.dli-title-metadata-item")
                for item in metadata_items:
                    text = item.text.strip()
                    if 'h' in text or 'm' in text:  # Detect duration like "2h 1m"
                        duration = text
                    elif text.isdigit() and len(text) == 4:  # Detect year like "2024"
                        year = text
            except:
                pass
    


            if [title, desc, rating, votes, image_url, duration, year] not in movie_data:
                movie_data.append([title, desc, rating, votes, image_url, duration, year])

        except Exception as e:
            print(f"⚠️ Skipping a movie due to error: {str(e)[:100]}")

def click_load_more():
    """Click the 'Load More' button if present."""
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        
        load_more_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.ipc-see-more__button'))
        )
        driver.execute_script("arguments[0].click();", load_more_button)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item:last-child'))
        )
        time.sleep(2)
        return True
    except (NoSuchElementException, TimeoutException):
        print("🔚 No more 'Load More' button or timeout.")
        return False
    except Exception as e:
        print(f"⚠️ Error clicking 'Load More': {str(e)[:100]}")
        return False

# Main execution
try:
    driver.get(url)
    time.sleep(3)

    while len(movie_data) < target_movies:
        prev = len(movie_data)
        scrape_current_page()
        new = len(movie_data) - prev
        print(f"✅ Scraped {new} new movies. Total: {len(movie_data)}")
        
        if len(movie_data) >= target_movies:
            break

        if not click_load_more():
            break

except Exception as e:
    print(f"❌ Error during scraping: {str(e)}")

finally:
    with open('imdb_2024_movies.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Movie Name', 'Storyline', 'Rating', 'Voting Count', 'Image URL', 'Duration', 'Year'])
        writer.writerows(movie_data)

    print(f"Saved data to imdb_2024_movies.csv with {len(movie_data)} entries.")
    driver.quit()

✅ Scraped 50 new movies. Total: 50
✅ Scraped 50 new movies. Total: 100
✅ Scraped 50 new movies. Total: 150
✅ Scraped 50 new movies. Total: 200
✅ Scraped 50 new movies. Total: 250
✅ Scraped 50 new movies. Total: 300
✅ Scraped 0 new movies. Total: 300
✅ Scraped 50 new movies. Total: 350
✅ Scraped 50 new movies. Total: 400
✅ Scraped 50 new movies. Total: 450
✅ Scraped 0 new movies. Total: 450
✅ Scraped 50 new movies. Total: 500
✅ Scraped 50 new movies. Total: 550
✅ Scraped 50 new movies. Total: 600
✅ Scraped 0 new movies. Total: 600
✅ Scraped 100 new movies. Total: 700
✅ Scraped 0 new movies. Total: 700
✅ Scraped 100 new movies. Total: 800
✅ Scraped 50 new movies. Total: 850
✅ Scraped 50 new movies. Total: 900
✅ Scraped 50 new movies. Total: 950
✅ Scraped 50 new movies. Total: 1000
✅ Scraped 50 new movies. Total: 1050
✅ Scraped 50 new movies. Total: 1100
✅ Scraped 50 new movies. Total: 1150
✅ Scraped 50 new movies. Total: 1200
Saved data to imdb_2024_movies.csv with 1200 entries.


In [1]:
import pandas as pd

data = pd.read_csv('imdb_2024_movies.csv')
print(data.head())  

          Movie Name                                          Storyline  \
0   We Bury the Dead  After a catastrophic military disaster, the de...   
1      The Substance  A fading celebrity takes a black-market drug: ...   
2  The Life of Chuck  A life-affirming, genre-bending story about th...   
3               Eden  Based on a factual account of a group of outsi...   
4   I Was a Stranger  Five strangers are pulled together, colliding ...   

   Rating Voting Count                                          Image URL  \
0     5.6        (7.6K  https://m.media-amazon.com/images/M/MV5BNzQwZD...   
1     7.2        (405K  https://m.media-amazon.com/images/M/MV5BZDQ1NG...   
2     7.3         (60K  https://m.media-amazon.com/images/M/MV5BZDhlMz...   
3     6.5         (40K  https://m.media-amazon.com/images/M/MV5BMmU3Yj...   
4     8.7         (11K  https://m.media-amazon.com/images/M/MV5BZGEwOT...   

  Duration    Year  
0   1h 34m  2024.0  
1   2h 21m  2024.0  
2   1h 51m  2024.0  
3 

In [2]:
data.shape

(1200, 7)

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Movie Name    1200 non-null   object 
 1   Storyline     1195 non-null   object 
 2   Rating        1197 non-null   float64
 3   Voting Count  1197 non-null   object 
 4   Image URL     1198 non-null   object 
 5   Duration      1186 non-null   object 
 6   Year          1191 non-null   float64
dtypes: float64(2), object(5)
memory usage: 65.8+ KB


In [4]:
data.isnull().sum()

Movie Name       0
Storyline        5
Rating           3
Voting Count     3
Image URL        2
Duration        14
Year             9
dtype: int64

Data Processing

In [5]:
import nltk
nltk.data.path.append("C:/Users/MUAFIQUA/AppData/Local/nltk_data")  # Your path from the error message

# Verify the path
print(nltk.data.path)

['C:\\Users\\MUAFIQUA/nltk_data', 'd:\\2\\5\\.venv\\nltk_data', 'd:\\2\\5\\.venv\\share\\nltk_data', 'd:\\2\\5\\.venv\\lib\\nltk_data', 'C:\\Users\\MUAFIQUA\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'C:/Users/MUAFIQUA/AppData/Local/nltk_data']


In [6]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
print(stopwords.words('english')[:10])


['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MUAFIQUA/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import pandas as pd
import re
import nltk
import os
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 1. NLTK Nuclear Option - Download EVERYTHING properly
def nuclear_nltk_download():
    print("Performing complete NLTK setup...")
    nltk_dir = os.path.join(os.path.expanduser("~"), "nltk_data")
    os.makedirs(nltk_dir, exist_ok=True)
    
    # Set download directory
    nltk.download('popular', download_dir=nltk_dir)
    nltk.download('punkt', download_dir=nltk_dir)
    nltk.download('stopwords', download_dir=nltk_dir)
    nltk.download('wordnet', download_dir=nltk_dir)
    nltk.download('omw-1.4', download_dir=nltk_dir)  # Required for WordNet
    nltk.download('punkt_tab', download_dir=nltk_dir)  # Specific missing resource
    
    # Refresh paths
    nltk.data.path.append(nltk_dir)
    print("NLTK setup complete. Resources available at:", nltk_dir)

# Execute the nuclear option
nuclear_nltk_download()

# 2. Verify all resources are accessible
def verify_nltk():
    try:
        word_tokenize("test")  # Verify punkt
        stopwords.words('english')  # Verify stopwords
        WordNetLemmatizer()  # Verify wordnet
        return True
    except LookupError as e:
        print(f"Missing resource: {e}")
        return False

if not verify_nltk():
    print("Critical NLTK resources missing. Trying manual fix...")
    nuclear_nltk_download()
    if not verify_nltk():
        raise EnvironmentError("Failed to setup NLTK. Please check internet connection.")

# 3. Load and preprocess data
try:
    data = pd.read_csv('imdb_movies_2024.csv', on_bad_lines='skip', encoding='latin1')
except Exception as e:
    print(f"Error loading CSV: {e}")
    exit()

# 4. Text processing with ultimate safeguards
def clean_text(text):
    try:
        if not isinstance(text, str) or not text.strip():
            return ""
        
        text = text.lower()  # Convert to lowercase
        text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphabetic characters
        
        # Tokenize with fallback
        try:
            tokens = word_tokenize(text)
        except:
            nuclear_nltk_download()
            tokens = word_tokenize(text)
            
        # Stopwords with fallback
        try:
            stop_words = set(stopwords.words('english'))
        except:
            nuclear_nltk_download()
            stop_words = set(stopwords.words('english'))
            
        tokens = [word for word in tokens if word not in stop_words]
        
        # Lemmatization with fallback
        try:
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(word) for word in tokens]
        except:
            nuclear_nltk_download()
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(word) for word in tokens]
            
        # Capitalize the first letter of each sentence
        sentences = text.split('. ')
        sentences = [sentence.capitalize() for sentence in sentences]  # Capitalize first letter of each sentence
        cleaned_text = '. '.join(sentences)
        
        return cleaned_text
    except Exception as e:
        print(f"Error processing text: {str(e)[:100]}...")
        return ""

# 5. Process data with progress feedback
print("\nStarting text processing...")
data['Cleaned_Storyline'] = data['Storyline'].fillna('').apply(clean_text)

# 6. Verify and save results
print("\nProcessing complete. Sample results:")
print(data[['Movie Name', 'Storyline', 'Cleaned_Storyline']].head(3))
data.to_csv('cleaned_movies.csv', index=False)
print(f"\nSuccessfully processed {len(data)} records. Saved to 'cleaned_movies.csv'")


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package 

Performing complete NLTK setup...


[nltk_data]    |   Package wordnet_ic is already up-to-date!
[nltk_data]    | Downloading package words to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package words is already up-to-date!
[nltk_data]    | Downloading package maxent_ne_chunker to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package maxent_ne_chunker is already up-to-date!
[nltk_data]    | Downloading package punkt to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package punkt is already up-to-date!
[nltk_data]    | Downloading package snowball_data to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package snowball_data is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\MUAFIQUA\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | 
[nltk_data]  Done downloading collection popu

NLTK setup complete. Resources available at: C:\Users\MUAFIQUA\nltk_data
Error loading CSV: [Errno 2] No such file or directory: 'imdb_movies_2024.csv'

Starting text processing...

Processing complete. Sample results:
          Movie Name                                          Storyline  \
0   We Bury the Dead  After a catastrophic military disaster, the de...   
1      The Substance  A fading celebrity takes a black-market drug: ...   
2  The Life of Chuck  A life-affirming, genre-bending story about th...   

                                   Cleaned_Storyline  
0  After a catastrophic military disaster the dea...  
1  A fading celebrity takes a blackmarket drug a ...  
2  A lifeaffirming genrebending story about three...  

Successfully processed 1200 records. Saved to 'cleaned_movies.csv'


: 

In [1]:
import pandas as pd
df=pd.read_csv("cleaned_movies.csv")

In [2]:
df

,Movie Name,Storyline,Rating,Voting Count,Image URL,Duration,Year,Cleaned_Storyline
0,We Bury the Dead,"After a catastrophic military disaster, the de...",5.6,(7.6K,https://m.media-amazon.com/images/M/MV5BNzQwZD...,1h 34m,2024.0,After a catastrophic military disaster the dea...
1,The Substance,A fading celebrity takes a black-market drug: ...,7.2,(405K,https://m.media-amazon.com/images/M/MV5BZDQ1NG...,2h 21m,2024.0,A fading celebrity takes a blackmarket drug a ...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",7.3,(60K,https://m.media-amazon.com/images/M/MV5BZDhlMz...,1h 51m,2024.0,A lifeaffirming genrebending story about three...
3,Eden,Based on a factual account of a group of outsi...,6.5,(40K,https://m.media-amazon.com/images/M/MV5BMmU3Yj...,2h 9m,2024.0,Based on a factual account of a group of outsi...
4,I Was a Stranger,"Five strangers are pulled together, colliding ...",8.7,(11K,https://m.media-amazon.com/images/M/MV5BZGEwOT...,1h 44m,2024.0,Five strangers are pulled together colliding o...
...,...,...,...,...,...,...,...,...
1195,Anunnakiler,"In Istanbul, the creator of a powerful aphrodi...",2.5,(518,https://m.media-amazon.com/images/M/MV5BMGEwMD...,1h 16m,2024.0,In istanbul the creator of a powerful aphrodis...
1196,Saint Clare,In a small town a solitary woman is haunted by...,4.6,(1.2K,https://m.media-amazon.com/images/M/MV5BMTlhOT...,1h 32m,2024.0,In a small town a solitary woman is haunted by...
1197,Dead Sea,Stranded in the open sea after a fatal acciden...,5.2,(6K,https://m.media-amazon.com/images/M/MV5BNGZiOG...,1h 28m,2024.0,Stranded in the open sea after a fatal acciden...
1198,Ardaas Sarbat De Bhalle Di,Gurmukh Singh and diverse individuals embark o...,8.0,(456,https://m.media-amazon.com/images/M/MV5BNDBhMW...,2h 13m,2024.0,Gurmukh singh and diverse individuals embark o...


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle


# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)  # Limit features to 5000 most frequent words
df['Cleaned_Storyline'] = df['Cleaned_Storyline'].fillna('')

# Fit and transform the cleaned storylines
tfidf_matrix = tfidf.fit_transform(df['Cleaned_Storyline'])

# Save TF-IDF vectorizer model and matrix to files
pickle.dump(tfidf, open('tfidf_model.pkl', 'wb'))
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))

print("TF-IDF model and matrix saved successfully!")

TF-IDF model and matrix saved successfully!


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Save similarity matrix
pickle.dump(cosine_sim, open('cosine_sim.pkl', 'wb'))

In [5]:
def recommend_movies(movie_name, cosine_sim_matrix, df, top_n=5):
    """
    Get top N similar movies based on cosine similarity of storylines.
    
    Args:
        movie_name (str): Name of the movie to find similar movies for
        cosine_sim_matrix (np.array): Precomputed cosine similarity matrix
        df (pd.DataFrame): DataFrame containing movie data
        top_n (int): Number of recommendations to return
        
    Returns:
        pd.DataFrame: Recommended movies or None if input movie not found
    """
    # Find movie indices (case-insensitive and partial match)
    matches = df[df['Movie Name'].str.contains(movie_name, case=False, regex=False)]
    
    if len(matches) == 0:
        print(f"Movie '{movie_name}' not found in dataset. Try another title.")
        print("Some available movies:", df['Movie Name'].head(10).tolist())
        return None
    
    # Use first match if multiple found
    movie_index = matches.index[0]
    
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim_matrix[movie_index]))
    
    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top N similar movies (skip the movie itself)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    return df.iloc[movie_indices][['Movie Name', 'Storyline', 'Cleaned_Storyline']]

In [6]:
# Try with different variations until you find a match
recommendations = recommend_movies("Nightbitch", cosine_sim, df)  # Case-insensitive
if recommendations is not None:
    print(recommendations)


            Movie Name                                          Storyline  \
181               MadS  A teenager stops off to see his dealer to test...   
1030          Upstream  Gao Zhilei's impulsive decision to become a st...   
472     Scorched Earth  Twelve years after he fled, career criminal Tr...   
455   Winter in Sokcho  A young Korean girl's life takes an unexpected...   
497       Take My Hand  At the peak of her career in London, an Austra...   

                                      Cleaned_Storyline  
181   A teenager stops off to see his dealer to test...  
1030  Gao zhileis impulsive decision to become a sta...  
472   Twelve years after he fled career criminal tro...  
455   A young korean girls life takes an unexpected ...  
497   At the peak of her career in london an austral...  


In [7]:
# Check exact movie names in your dataset
print(df['Movie Name'].unique())

# Search for partial matches
print(df[df['Movie Name'].str.contains('incep', case=False)])

['We Bury the Dead' 'The Substance' 'The Life of Chuck' ... 'Dead Sea'
 'Ardaas Sarbat De Bhalle Di' 'Kill Em All 2']
Empty DataFrame
Columns: [Movie Name, Storyline, Rating, Voting Count, Image URL, Duration, Year, Cleaned_Storyline]
Index: []
